In [ ]:
# Module 3 - Lab 1: Vision Transformers in Keras

import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflow_addons as tfa  # pip install tensorflow-addons
import numpy as np

# Simple Vision Transformer Block
class ViTBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim, dropout=dropout)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = models.Sequential([
            layers.Dense(mlp_dim, activation=tf.nn.gelu),
            layers.Dropout(dropout),
            layers.Dense(embed_dim),
            layers.Dropout(dropout)
        ])
    
    def call(self, x, training=False):
        # Attention block
        x_norm = self.norm1(x)
        attn_output = self.att(x_norm, x_norm)
        x = x + attn_output
        
        # MLP block
        x_norm = self.norm2(x)
        mlp_output = self.mlp(x_norm, training=training)
        return x + mlp_output

# Full Vision Transformer Model
def create_vit_model(input_shape=(224, 224, 3), num_classes=10, 
                     patch_size=16, embed_dim=192, depth=8, num_heads=3):
    
    inputs = layers.Input(shape=input_shape)
    
    # Patch Embedding
    patches = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding='valid')(inputs)
    patches = layers.Reshape((-1, embed_dim))(patches)
    
    # Class Token + Positional Embedding
    batch_size = tf.shape(patches)[0]
    cls_token = tf.zeros((batch_size, 1, embed_dim))  # Learnable class token
    cls_token = layers.Dense(embed_dim)(cls_token)   # Make it learnable
    patches = tf.concat([cls_token, patches], axis=1)
    
    positions = tf.range(1 + patches.shape[1])
    pos_embedding = layers.Embedding(input_dim=1 + patches.shape[1], output_dim=embed_dim)(positions)
    patches = patches + pos_embedding
    
    # Transformer Blocks
    for _ in range(depth):
        patches = ViTBlock(embed_dim, num_heads, mlp_dim=embed_dim*4)(patches)
    
    # Classification Head
    x = layers.LayerNormalization(epsilon=1e-6)(patches[:, 0])  # Use class token
    x = layers.Dense(512, activation='gelu')(x)
    x = layers.Dropout(0.1)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = models.Model(inputs, outputs)
    return model

model = create_vit_model(num_classes=5)  # Change num_classes to your dataset
model.compile(optimizer=tfa.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# Module 3 - Lab 2: Vision Transformers in PyTorch

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

class PatchEmbedding(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=192):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.proj(x)          # (B, embed_dim, H, W)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)
        return x

class ViT(nn.Module):
    def __init__(self, img_size=224, patch_size=16, embed_dim=192, depth=8, 
                 num_heads=3, num_classes=5):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=embed_dim, nhead=num_heads, dim_feedforward=embed_dim*4,
                activation='gelu', batch_first=True, dropout=0.1
            ) for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        return self.head(x[:, 0])   # Class token output

# Usage
model = ViT(num_classes=5)   # Change to your number of classes
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Example training setup (same as previous PyTorch lab)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)

In [ ]:
# Module 3 - Lab 3: CNN + Transformer Hybrid for Land Classification

import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_Transformer_Hybrid(nn.Module):
    def __init__(self, num_classes=6):  # e.g., Forest, Desert, Water, Urban, etc.
        super().__init__()
        
        # CNN Backbone (Feature Extractor)
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((14, 14))   # Fixed size for transformer
        )
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256, nhead=8, dim_feedforward=1024, 
            dropout=0.1, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.cnn(x)                    # (B, 256, 14, 14)
        B, C, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)   # (B, H*W, C) for transformer
        x = self.transformer(x)
        x = x.transpose(1, 2).reshape(B, C, H, W)
        return self.classifier(x)

# Evaluation & Comparison
def evaluate_model(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

print("Hybrid CNN-Transformer model ready for Land Classification!")